In [3]:
!pip install yfinance pandas numpy

In [9]:
# stocks.py

LARGE_CAP = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "SBIN.NS", "BHARTIARTL.NS", "ITC.NS", "KOTAKBANK.NS",
    "LT.NS", "AXISBANK.NS", "ASIANPAINT.NS", "MARUTI.NS", "SUNPHARMA.NS",
    "TITAN.NS", "BAJFINANCE.NS", "WIPRO.NS", "NESTLEIND.NS", "ULTRACEMCO.NS",
    "POWERGRID.NS", "NTPC.NS", "TECHM.NS", "HCLTECH.NS", "ONGC.NS",
    "COALINDIA.NS", "BAJAJFINSV.NS", "TATAMOTORS.NS", "ADANIENT.NS", "ADANIPORTS.NS",
    "DIVISLAB.NS", "DRREDDY.NS", "EICHERMOT.NS", "GRASIM.NS", "HEROMOTOCO.NS",
    "INDUSINDBK.NS", "JSWSTEEL.NS", "M&M.NS", "SBILIFE.NS", "TATASTEEL.NS",
    "CIPLA.NS", "BPCL.NS", "BRITANNIA.NS", "APOLLOHOSP.NS", "BAJAJ-AUTO.NS",
    "HDFCLIFE.NS", "HINDALCO.NS", "TATACONSUM.NS", "UPL.NS", "VEDL.NS",
    # Add more to reach 100...
]

MID_CAP = [
    "PERSISTENT.NS", "MPHASIS.NS", "LTIM.NS", "COFORGE.NS", "ZOMATO.NS",
    "NYKAA.NS", "DMART.NS", "IRCTC.NS", "HAPPSTMNDS.NS", "TANLA.NS",
    "CAMS.NS", "CDSL.NS", "MCX.NS", "POLYCAB.NS", "SCHAEFFLER.NS",
    "SOLARINDS.NS", "TRENTLTD.NS", "VOLTAS.NS", "WHIRLPOOL.NS", "AUROPHARMA.NS",
    "BALKRISIND.NS", "BATAINDIA.NS", "BERGEPAINT.NS", "CANFINHOME.NS", "CHOLAFIN.NS",
    "CROMPTON.NS", "CUMMINSIND.NS", "DEEPAKNTR.NS", "DIXON.NS", "ESCORTS.NS",
    "GODREJPROP.NS", "GRANULES.NS", "GUJGASLTD.NS", "HDFCAMC.NS", "HINDPETRO.NS",
    "IDFCFIRSTB.NS", "INDHOTEL.NS", "INDUSTOWER.NS", "INTELLECT.NS", "KANSAINER.NS",
    "LUPIN.NS", "MFSL.NS", "MINDTREE.NS", "MOTILALOFS.NS", "MRF.NS",
    "NMDC.NS", "OBEROIRLTY.NS", "PAGEIND.NS", "PETRONET.NS", "PNB.NS",
    # Add more to reach 150...
]

SMALL_CAP = [
    "BBOX.NS", "TEJASNET.NS", "RAILTEL.NS", "IRFC.NS", "RVNL.NS",
    "NAZARA.NS", "HAPPYFORGE.NS", "MAZDOCK.NS", "COCHINSHIP.NS", "GRSE.NS",
    "BEML.NS", "BEL.NS", "HAL.NS", "MIDHANI.NS", "ASTRAL.NS",
    "ROUTE.NS", "CAMPUS.NS", "SENCO.NS", "SAPPHIRE.NS", "BALAMINES.NS",
    "CHEMCON.NS", "FINEORG.NS", "GALAXYSURF.NS", "HFCL.NS", "IDEAFORGE.NS",
    "JYOTHYLAB.NS", "KFINTECH.NS", "LATENTVIEW.NS", "MEDPLUS.NS", "NETWEB.NS",
    "ORIENTELEC.NS", "POLICYBZR.NS", "RHIM.NS", "SBFC.NS", "SIGACHI.NS",
    "STLTECH.NS", "TANLA.NS", "TITAGARH.NS", "TTKPRESTIG.NS", "UNIPARTS.NS",
    # Add more to reach 100...
]

ALL_STOCKS = {
    "large": LARGE_CAP,
    "mid":   MID_CAP,
    "small": SMALL_CAP
}

In [5]:
import yfinance as yf
import pandas as pd
import os
import time

# ── Config ──────────────────────────────────────────
START_DATE  = "2021-01-01"
END_DATE    = "2025-01-01"
OUTPUT_DIR  = "data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Download Function ────────────────────────────────
def download_stock(ticker: str, cap_type: str) -> pd.DataFrame | None:
    """Download OHLCV for one ticker and tag it with cap_type."""
    try:
        df = yf.download(ticker, start=START_DATE, end=END_DATE,
                         auto_adjust=True, progress=False)
        if df.empty or len(df) < 100:          # skip tickers with too little data
            print(f"  ⚠️  Skipped {ticker} — insufficient data")
            return None

        df.reset_index(inplace=True)
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        df["Ticker"]   = ticker
        df["Cap_Type"] = cap_type
        return df

    except Exception as e:
        print(f"  ❌  Error downloading {ticker}: {e}")
        return None

# ── Main Download Loop ───────────────────────────────
all_dfs   = []
failed    = []

for cap_type, tickers in ALL_STOCKS.items():
    print(f"\n{'='*50}")
    print(f"  Downloading {cap_type.upper()} CAP stocks ({len(tickers)} tickers)")
    print(f"{'='*50}")

    cap_dfs = []
    for i, ticker in enumerate(tickers, 1):
        print(f"  [{i}/{len(tickers)}] {ticker}", end="  ")
        df = download_stock(ticker, cap_type)
        if df is not None:
            cap_dfs.append(df)
            print(f"✅  {len(df)} rows")
        else:
            failed.append(ticker)
        time.sleep(0.3)     # polite delay to avoid rate-limiting

    # Save per-cap CSV
    if cap_dfs:
        cap_df = pd.concat(cap_dfs, ignore_index=True)
        path   = os.path.join(OUTPUT_DIR, f"{cap_type}_cap_raw.csv")
        cap_df.to_csv(path, index=False)
        print(f"\n  💾  Saved {len(cap_df):,} rows → {path}")
        all_dfs.append(cap_df)

# ── Master CSV ───────────────────────────────────────
if all_dfs:
    master = pd.concat(all_dfs, ignore_index=True)
    master.to_csv(os.path.join(OUTPUT_DIR, "master_raw.csv"), index=False)
    print(f"\n✅  Master file saved: {len(master):,} rows, {master['Ticker'].nunique()} tickers")

# ── Failed Tickers Report ────────────────────────────
if failed:
    print(f"\n⚠️  Failed tickers ({len(failed)}): {failed}")


  [1/50] RELIANCE.NS  ✅  987 rows
  [2/50] TCS.NS  ✅  987 rows
  [3/50] HDFCBANK.NS  ✅  987 rows
  [4/50] INFY.NS  ✅  987 rows
  [5/50] ICICIBANK.NS  ✅  987 rows
  [6/50] HINDUNILVR.NS  ✅  987 rows
  [7/50] SBIN.NS  ✅  987 rows
  [8/50] BHARTIARTL.NS  ✅  987 rows
  [9/50] ITC.NS  ✅  987 rows
  [10/50] KOTAKBANK.NS  ✅  987 rows
  [11/50] LT.NS  ✅  987 rows
  [12/50] AXISBANK.NS  ✅  987 rows
  [13/50] ASIANPAINT.NS  ✅  987 rows
  [14/50] MARUTI.NS  ✅  987 rows
  [15/50] SUNPHARMA.NS  ✅  987 rows
  [16/50] TITAN.NS  ✅  987 rows
  [17/50] BAJFINANCE.NS  ✅  987 rows
  [18/50] WIPRO.NS  ✅  987 rows
  [19/50] NESTLEIND.NS  ✅  987 rows
  [20/50] ULTRACEMCO.NS  ✅  987 rows
  [21/50] POWERGRID.NS  ✅  987 rows
  [22/50] NTPC.NS  ✅  987 rows
  [23/50] TECHM.NS  ✅  987 rows
  [24/50] HCLTECH.NS  ✅  987 rows
  [25/50] ONGC.NS  ✅  987 rows
  [26/50] COALINDIA.NS  ✅  987 rows
  [27/50] BAJAJFINSV.NS  ✅  987 rows
  [28/50] TATAMOTORS.NS  

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


  ⚠️  Skipped TATAMOTORS.NS — insufficient data
  [29/50] ADANIENT.NS  ✅  987 rows
  [30/50] ADANIPORTS.NS  ✅  987 rows
  [31/50] DIVISLAB.NS  ✅  987 rows
  [32/50] DRREDDY.NS  ✅  987 rows
  [33/50] EICHERMOT.NS  ✅  987 rows
  [34/50] GRASIM.NS  ✅  987 rows
  [35/50] HEROMOTOCO.NS  ✅  987 rows
  [36/50] INDUSINDBK.NS  ✅  987 rows
  [37/50] JSWSTEEL.NS  ✅  987 rows
  [38/50] M&M.NS  ✅  987 rows
  [39/50] SBILIFE.NS  ✅  987 rows
  [40/50] TATASTEEL.NS  ✅  987 rows
  [41/50] CIPLA.NS  ✅  987 rows
  [42/50] BPCL.NS  ✅  987 rows
  [43/50] BRITANNIA.NS  ✅  987 rows
  [44/50] APOLLOHOSP.NS  ✅  987 rows
  [45/50] BAJAJ-AUTO.NS  ✅  987 rows
  [46/50] HDFCLIFE.NS  ✅  987 rows
  [47/50] HINDALCO.NS  ✅  987 rows
  [48/50] TATACONSUM.NS  ✅  987 rows
  [49/50] UPL.NS  ✅  987 rows
  [50/50] VEDL.NS  ✅  987 rows

  💾  Saved 48,363 rows → data/raw\large_cap_raw.csv

  [1/50] PERSISTENT.NS  ✅  987 rows
  [2/50] MPHASIS.NS  ✅  987 rows
  [3/50] LTIM.NS  

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LTIM.NS"}}}
$LTIM.NS: possibly delisted; no timezone found

1 Failed download:
['LTIM.NS']: possibly delisted; no timezone found


  ⚠️  Skipped LTIM.NS — insufficient data
  [4/50] COFORGE.NS  ✅  987 rows
  [5/50] ZOMATO.NS  

...

  ⚠️  Skipped ZOMATO.NS — insufficient data
  [6/50] NYKAA.NS  ✅  776 rows
  [7/50] DMART.NS  ✅  987 rows
  [8/50] IRCTC.NS  ✅  987 rows
  [9/50] HAPPSTMNDS.NS  ✅  987 rows
  [10/50] TANLA.NS  ✅  987 rows
  [11/50] CAMS.NS  ✅  987 rows
  [12/50] CDSL.NS  ✅  987 rows
  [13/50] MCX.NS  ✅  987 rows
  [14/50] POLYCAB.NS  ✅  987 rows
  [15/50] SCHAEFFLER.NS  ✅  987 rows
  [16/50] SOLARINDS.NS  ✅  987 rows
  [17/50] TRENTLTD.NS  

$TRENTLTD.NS: possibly delisted; no timezone found

1 Failed download:
['TRENTLTD.NS']: possibly delisted; no timezone found


  ⚠️  Skipped TRENTLTD.NS — insufficient data
  [18/50] VOLTAS.NS  ✅  987 rows
  [19/50] WHIRLPOOL.NS  ✅  987 rows
  [20/50] AUROPHARMA.NS  ✅  987 rows
  [21/50] BALKRISIND.NS  ✅  987 rows
  [22/50] BATAINDIA.NS  ✅  987 rows
  [23/50] BERGEPAINT.NS  ✅  987 rows
  [24/50] CANFINHOME.NS  ✅  987 rows
  [25/50] CHOLAFIN.NS  ✅  987 rows
  [26/50] CROMPTON.NS  ✅  987 rows
  [27/50] CUMMINSIND.NS  ✅  987 rows
  [28/50] DEEPAKNTR.NS  ✅  987 rows
  [29/50] DIXON.NS  ✅  987 rows
  [30/50] ESCORTS.NS  ✅  987 rows
  [31/50] GODREJPROP.NS  ✅  987 rows
  [32/50] GRANULES.NS  ✅  987 rows
  [33/50] GUJGASLTD.NS  ✅  987 rows
  [34/50] HDFCAMC.NS  ✅  987 rows
  [35/50] HINDPETRO.NS  ✅  987 rows
  [36/50] IDFCFIRSTB.NS  ✅  987 rows
  [37/50] INDHOTEL.NS  ✅  987 rows
  [38/50] INDUSTOWER.NS  ✅  987 rows
  [39/50] INTELLECT.NS  ✅  987 rows
  [40/50] KANSAINER.NS  ✅  987 rows
  [41/50] LUPIN.NS  ✅  987 rows
  [42/50] MFSL.NS  ✅  987 rows
  [43/50] MINDTREE.NS  

$MINDTREE.NS: possibly delisted; no timezone found

1 Failed download:
['MINDTREE.NS']: possibly delisted; no timezone found


  ⚠️  Skipped MINDTREE.NS — insufficient data
  [44/50] MOTILALOFS.NS  ✅  987 rows
  [45/50] MRF.NS  ✅  987 rows
  [46/50] NMDC.NS  ✅  987 rows
  [47/50] OBEROIRLTY.NS  ✅  987 rows
  [48/50] PAGEIND.NS  ✅  987 rows
  [49/50] PETRONET.NS  ✅  987 rows
  [50/50] PNB.NS  ✅  987 rows

  💾  Saved 45,191 rows → data/raw\mid_cap_raw.csv

  [1/40] BBOX.NS  ✅  987 rows
  [2/40] TEJASNET.NS  ✅  987 rows
  [3/40] RAILTEL.NS  ✅  948 rows
  [4/40] IRFC.NS  ✅  968 rows
  [5/40] RVNL.NS  ✅  987 rows
  [6/40] NAZARA.NS  ✅  928 rows
  [7/40] HAPPYFORGE.NS  ✅  249 rows
  [8/40] MAZDOCK.NS  ✅  987 rows
  [9/40] COCHINSHIP.NS  ✅  987 rows
  [10/40] GRSE.NS  ✅  987 rows
  [11/40] BEML.NS  ✅  987 rows
  [12/40] BEL.NS  ✅  987 rows
  [13/40] HAL.NS  ✅  987 rows
  [14/40] MIDHANI.NS  ✅  987 rows
  [15/40] ASTRAL.NS  ✅  987 rows
  [16/40] ROUTE.NS  ✅  987 rows
  [17/40] CAMPUS.NS  ✅  655 rows
  [18/40] SENCO.NS  ✅  360 rows
  [19/40] SAPPHIRE.NS  ✅  770 rows
  [20/40] BALAMINES.NS  ✅  987 rows
  [21/40] CHEMCON

In [20]:
import pandas as pd

master = pd.read_csv(
    "data/raw/master_raw.csv",
    parse_dates=["index"]
)

master.rename(columns={"index": "Date"}, inplace=True)

print("=== MASTER DATASET OVERVIEW ===")
print(f"Shape          : {master.shape}")
print(f"Date range     : {master['Date'].min()} → {master['Date'].max()}")
print(f"Total tickers  : {master['Ticker'].nunique()}")
print(f"\nStocks per cap type:")
print(master.groupby("Cap_Type")["Ticker"].nunique())
print(f"\nMissing values :")
print(master.isnull().sum())
print(f"\nSample rows:")
master.head()

=== MASTER DATASET OVERVIEW ===
Shape          : (123281, 8)
Date range     : 2021-01-01 00:00:00 → 2024-12-31 00:00:00
Total tickers  : 130

Stocks per cap type:
Cap_Type
large    45
mid      46
small    40
Name: Ticker, dtype: int64

Missing values :
Date        0
Close       0
High        0
Low         0
Open        0
Volume      0
Ticker      0
Cap_Type    0
dtype: int64

Sample rows:


,Date,Close,High,Low,Open,Volume,Ticker,Cap_Type
0,2021-01-01,901.194641,905.502206,898.700750,901.421320,10015175,RELIANCE.NS,large
1,2021-01-04,902.713623,906.363699,892.352703,904.640696,24513534,RELIANCE.NS,large
2,2021-01-05,891.491150,899.426182,886.911450,892.806069,24123091,RELIANCE.NS,large
3,2021-01-06,867.980774,891.445793,863.854552,891.400457,46401468,RELIANCE.NS,large
4,2021-01-07,866.575195,881.923840,863.854624,870.814757,32325918,RELIANCE.NS,large
